# TP02 — Procesos ETL (pandas)
   Bases de Datos Masivas (11088) · Gustavo An Contardi - 182818
   

## 0. Extracción — lectura del dataset
> A partir del dataset Education Data del World Bank Group, desarrollar los flujos
> necesarios para generar los siguientes archivos de salida (en formato CSV).

In [1]:
import re
from pathlib import Path

import pandas as pd

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)

# El archivo conserva el nombre con el que lo descarga el World Bank. Lo dejo en una
# constante para no repetir el número de versión en cada lectura.
ARCHIVO = "data/API_4_DS2_en_csv_v2_3434744.csv"

# Los tres CSV del TP van a una carpeta aparte, así no se mezclan con la entrada
# y después puedo comparar contra lo que genere Apache Hop.
SALIDA = Path("salida")
SALIDA.mkdir(exist_ok=True)

In [2]:
crudo = pd.read_csv(ARCHIVO)

print("filas x columnas:", crudo.shape)
print("primeras 6 columnas:", list(crudo.columns[:6]))
print("ultimas 4 columnas: ", list(crudo.columns[-4:]))

filas x columnas: (43624, 69)
primeras 6 columnas: ['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code', '1960', '1961']
ultimas 4 columnas:  ['2021', '2022', '2023', 'Unnamed: 68']


In [3]:
# Recorto a 8 columnas porque el dataset es ancho (un año por columna) y entero no entra.
crudo.iloc[:3, :8]

,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963
0,Aruba,ABW,Population ages 15-64 (% of total population),SP.POP.1564.TO.ZS,54.495678,54.588701,54.585630,54.674206
1,Aruba,ABW,Population ages 0-14 (% of total population),SP.POP.0014.TO.ZS,43.131043,42.949419,42.852732,42.661157
2,Aruba,ABW,"Unemployment, total (% of total labor force) (...",SL.UEM.TOTL.ZS,NaN,NaN,NaN,NaN


In [4]:
BASE = ["Country Name", "Country Code", "Indicator Name", "Indicator Code"]

# Detecto los años por el nombre de la columna en vez de escribir el rango a mano:
# si el World Bank publica 2024, esto sigue funcionando sin tocar nada.
ANIOS = [c for c in crudo.columns if c.isdigit()]

SOBRAN = [c for c in crudo.columns if c not in BASE and c not in ANIOS]

print("columnas base :", len(BASE))
print("columnas año  :", len(ANIOS), f"({ANIOS[0]} a {ANIOS[-1]})")
print("columnas que sobran:", SOBRAN)

columnas base : 4
columnas año  : 64 (1960 a 2023)
columnas que sobran: ['Unnamed: 68']


In [5]:
# Me quedo con las columnas que declaré arriba. Sacar "Unnamed: 68" es la consecuencia,
# no el objetivo: lo que quiero es que el esquema esperado quede escrito en el notebook.
wb = crudo[BASE + ANIOS].copy()

print("antes  :", crudo.shape)
print("despues:", wb.shape)

antes  : (43624, 69)
despues: (43624, 68)


In [6]:
print("nulos en los campos base:")
print(wb[BASE].isna().sum().to_string())
print()
print("filas totales        :", len(wb))
print("paises distintos     :", wb["Country Name"].nunique())
print("indicadores distintos:", wb["Indicator Name"].nunique())
print("paises x indicadores :", wb["Country Name"].nunique() * wb["Indicator Name"].nunique())

nulos en los campos base:
Country Name      0
Country Code      0
Indicator Name    0
Indicator Code    0

filas totales        : 43624
paises distintos     : 266
indicadores distintos: 164
paises x indicadores : 43624


El archivo trae 43.624 filas y 69 columnas. Las últimas 64 son un año cada una, de
1960 a 2023, así que el formato es ancho: una fila por combinación de país e
indicador, con la serie temporal desplegada a lo largo.

La columna 69 es `Unnamed: 68` y está vacía en las 43.624 filas. Aparece porque cada
línea del CSV termina en coma y pandas lee eso como un campo más. Lo resolví
declarando las columnas que espero (`BASE + ANIOS`) y quedándome con esas, en vez de
dropear la sobrante. Es una línea más de código, pero deja escrito cuál es el esquema
que el notebook espera del archivo: si el formato cambia, falla acá y no diez celdas
más abajo. Los años los detecto por el nombre de la columna en lugar de escribir el
rango a mano, así que si el World Bank publica 2024 no hay que tocar nada.

266 países por 164 indicadores da 43.624, exactamente el total de filas. La grilla
está completa: cada país tiene una fila por cada indicador, haya dato o no. Esto
importa para los dos primeros puntos del TP, porque significa que cada país aparece
repetido 164 veces y cada indicador 266 veces. Los duplicados que pide eliminar la
consigna no son un defecto del archivo, son la forma normal de este formato.

Los cuatro campos base no tienen ningún nulo. No lo leo como que el dataset esté
limpio, solo dice que las columnas identificatorias están completas. Los faltantes
están en las columnas de año, que es donde viven los datos: se ve en la fila de
`Unemployment` de Aruba, sin valores entre 1960 y 1963.

## 1. Salida A — CSV de Países
> **CSV de Países.** Campos requeridos: `id` (identificador numérico entero
> autoincremental), `nombre_pais` (nombre oficial del país o entidad regional),
> `codigo_pais` (código estándar de 3 letras, ISO Alpha-3).
> Transformaciones esperadas: limpieza de registros nulos o datos faltantes en los
> campos base, y eliminación de duplicados.

In [7]:
paises = wb[["Country Name", "Country Code"]].copy()
antes = len(paises)

# La consigna pide limpiar nulos en los campos base. Acá no hay ninguno, pero el paso
# va igual: un ETL no asume que la entrada viene limpia, la verifica y deja el número.
paises = paises.dropna(subset=["Country Name", "Country Code"])
sin_nulos = len(paises)

paises = paises.drop_duplicates()
sin_dup = len(paises)

print(f"filas de entrada      : {antes}")
print(f"tras descartar nulos  : {sin_nulos:>6}  (descartadas: {antes - sin_nulos})")
print(f"tras quitar duplicados: {sin_dup:>6}  (descartadas: {sin_nulos - sin_dup})")

filas de entrada      : 43624
tras descartar nulos  :  43624  (descartadas: 0)
tras quitar duplicados:    266  (descartadas: 43358)


In [8]:
paises = (
    paises
    # Ordeno explícito antes de numerar. El id depende del orden de las filas, así que si
    # no lo fijo acá, Hop me va a dar otra numeración y después no puedo comparar los CSV.
    .sort_values("Country Name", kind="stable")
    .reset_index(drop=True)
    .rename(columns={"Country Name": "nombre_pais", "Country Code": "codigo_pais"})
)

# id autoincremental desde 1, como pide la consigna.
paises.insert(0, "id", range(1, len(paises) + 1))

print(paises.shape)
paises.head()

(266, 3)


,id,nombre_pais,codigo_pais
0,1,Afghanistan,AFG
1,2,Africa Eastern and Southern,AFE
2,3,Africa Western and Central,AFW
3,4,Albania,ALB
4,5,Algeria,DZA


In [9]:
print("filas               :", len(paises))
print("nulos               :", int(paises.isna().sum().sum()))
print("nombres duplicados  :", int(paises["nombre_pais"].duplicated().sum()))
print("codigos duplicados  :", int(paises["codigo_pais"].duplicated().sum()))

# La consigna pide ISO Alpha-3. Lo verifico en vez de darlo por hecho.
print("codigos que no son 3 letras mayusculas:",
      int((~paises["codigo_pais"].str.fullmatch(r"[A-Z]{3}")).sum()))

print("id de", paises["id"].min(), "a", paises["id"].max(),
      "| sin huecos:", paises["id"].nunique() == len(paises) == paises["id"].max())

filas               : 266
nulos               : 0
nombres duplicados  : 0
codigos duplicados  : 0
codigos que no son 3 letras mayusculas: 0
id de 1 a 266 | sin huecos: True


In [10]:
destino = SALIDA / "paises.csv"
paises.to_csv(destino, index=False, encoding="utf-8")

# Releo el archivo escrito en vez de confiar en lo que tengo en memoria. Es la única
# forma de verificar que lo que entrego es lo que creo que generé.
control = pd.read_csv(destino)
print("escrito:", destino, "|", control.shape)
control.head()

escrito: salida/paises.csv | (266, 3)


,id,nombre_pais,codigo_pais
0,1,Afghanistan,AFG
1,2,Africa Eastern and Southern,AFE
2,3,Africa Western and Central,AFW
3,4,Albania,ALB
4,5,Algeria,DZA


De las 43.624 filas de entrada quedan 266. El `dropna` no descartó ninguna y el
`drop_duplicates` sacó 43.358. Es lo esperable: como cada país tiene una fila por cada
uno de los 164 indicadores, el par (nombre, código) viene repetido 164 veces. Los
duplicados no son datos sucios, son la forma en que este formato guarda la información.

Dejo el `dropna` aunque descarte cero filas, y verifico que los 266 códigos sean tres
letras mayúsculas aunque cumplan todos. En los dos casos el resultado limpio también es
información: queda escrito que lo chequeé en vez de suponerlo. Tampoco hay nombres ni
códigos repetidos, así que la correspondencia entre los dos campos es uno a uno.

El `id` lo asigno después de ordenar por `nombre_pais`. La numeración depende del orden
de las filas, así que si no lo fijo acá, el mismo país puede quedar con un id distinto
cuando repita el proceso en Apache Hop y después no voy a poder comparar los dos CSV.
Es la decisión de esta sección que más condiciona la verificación final.

Incluí las entidades regionales y no solo los países. La consigna las contempla cuando
pide el "nombre oficial del país o entidad regional", y separarlas necesitaría una lista
que este CSV no trae. Ya con los ids 2 y 3 aparecen `Africa Eastern and Southern` y
`Africa Western and Central`. Lo dejo anotado para la salida C: ahí armo un ranking por
duración de la primaria, y un agregado regional no es comparable con un país individual.

## 2. Salida B — CSV de Preguntas
> **CSV de Preguntas.** Campos requeridos: `id` (identificador numérico entero
> autoincremental), `pregunta` (descripción corta del indicador o pregunta educativa).
> Transformaciones esperadas: expresión regular (RegEx) o manipulación de cadenas para
> remover completamente cualquier contenido que se encuentre entre paréntesis
> (incluyendo los propios paréntesis) y aplicar un trim para eliminar espacios extra
> resultantes. Consolidar indicadores únicos sin duplicados.

In [11]:
# Los nombres de indicador son largos y pandas los corta en 50 caracteres. Sin esto no
# se ve el efecto de la regex, que es justamente lo que quiero mostrar.
pd.set_option("display.max_colwidth", 80)

# El patrón: un paréntesis que no contiene otro paréntesis adentro, con el espacio previo
# incluido en el match. Así el trim no tiene que arreglar dobles espacios después.
PARENTESIS = r"\s*\([^()]*\)"

indicadores = wb["Indicator Name"].dropna().drop_duplicates()

print("indicadores unicos en el archivo:", len(indicadores))
print()
print("cuantos parentesis tiene cada uno:")
print(indicadores.str.count(r"\(").value_counts().sort_index().to_string())

indicadores unicos en el archivo: 164

cuantos parentesis tiene cada uno:
Indicator Name
0     21
1    111
2     32


In [12]:
# Comparo contra la versión ingenua de la regex. El `.*` es codicioso: matchea desde el
# primer "(" hasta el último ")", así que en los nombres con dos paréntesis se lleva
# puesto el texto que hay en el medio, que no había que borrar.
codicioso = indicadores.str.replace(r"\s*\(.*\)", "", regex=True).str.strip()
correcto  = indicadores.str.replace(PARENTESIS, "", regex=True).str.strip()

comp = pd.DataFrame({"original": indicadores, "codicioso": codicioso, "correcto": correcto})
print("difieren en", int((comp.codicioso != comp.correcto).sum()), "casos")
comp[comp.codicioso != comp.correcto]

difieren en 5 casos


,original,codicioso,correcto
152,"School enrollment, tertiary (gross), gender parity index (GPI)","School enrollment, tertiary","School enrollment, tertiary, gender parity index"
153,"School enrollment, secondary (gross), gender parity index (GPI)","School enrollment, secondary","School enrollment, secondary, gender parity index"
154,"School enrollment, primary and secondary (gross), gender parity index (GPI)","School enrollment, primary and secondary","School enrollment, primary and secondary, gender parity index"
155,"School enrollment, primary (gross), gender parity index (GPI)","School enrollment, primary","School enrollment, primary, gender parity index"
162,"Literacy rate, youth (ages 15-24), gender parity index (GPI)","Literacy rate, youth","Literacy rate, youth, gender parity index"


In [13]:
# Al sacar los paréntesis, indicadores que eran distintos quedan con el mismo nombre.
# Antes de deduplicar quiero ver exactamente qué se va a fundir con qué.
limpios = pd.DataFrame({"original": indicadores, "pregunta": correcto})
colision = limpios[limpios["pregunta"].duplicated(keep=False)].sort_values("pregunta")

print(f"{len(colision)} indicadores colapsan en {colision['pregunta'].nunique()} preguntas")
colision

26 indicadores colapsan en 13 preguntas


,original,pregunta
15,"Government expenditure on education, total (% of GDP)","Government expenditure on education, total"
16,"Government expenditure on education, total (% of government expenditure)","Government expenditure on education, total"
132,"Primary education, pupils (% female)","Primary education, pupils"
133,"Primary education, pupils","Primary education, pupils"
101,"Primary education, teachers (% female)","Primary education, teachers"
102,"Primary education, teachers","Primary education, teachers"
124,"School enrollment, primary (% net)","School enrollment, primary"
130,"School enrollment, primary (% gross)","School enrollment, primary"
123,"School enrollment, primary, female (% net)","School enrollment, primary, female"
129,"School enrollment, primary, female (% gross)","School enrollment, primary, female"


In [14]:
preguntas = (
    correcto.drop_duplicates()
    # Mismo criterio que en países: ordeno antes de numerar para que Hop, que asigna la
    # secuencia según el orden de llegada de las filas, me dé la misma numeración.
    .sort_values(kind="stable")
    .reset_index(drop=True)
    .rename("pregunta")
    .to_frame()
)
preguntas.insert(0, "id", range(1, len(preguntas) + 1))

print(f"{len(indicadores)} indicadores -> {len(preguntas)} preguntas unicas")
preguntas.head()

164 indicadores -> 151 preguntas unicas


,id,pregunta
0,1,"Adjusted net enrollment rate, primary"
1,2,"Adjusted net enrollment rate, primary, female"
2,3,"Adjusted net enrollment rate, primary, male"
3,4,Adolescents out of school
4,5,"Adolescents out of school, female"


In [15]:
print("filas                       :", len(preguntas))
print("nulos                       :", int(preguntas.isna().sum().sum()))
print("preguntas duplicadas        :", int(preguntas["pregunta"].duplicated().sum()))

# Que no haya sobrevivido ningún paréntesis suelto ni espacio de más es lo que pide
# literalmente la consigna, así que lo verifico en vez de asumirlo.
print("con parentesis sobreviviente:", int(preguntas["pregunta"].str.contains(r"[()]").sum()))
print("con espacio al borde        :", int((preguntas["pregunta"] != preguntas["pregunta"].str.strip()).sum()))
print("con doble espacio interno   :", int(preguntas["pregunta"].str.contains("  ").sum()))
print("vacias                      :", int(preguntas["pregunta"].eq("").sum()))

print("id de", preguntas["id"].min(), "a", preguntas["id"].max(),
      "| sin huecos:", preguntas["id"].nunique() == len(preguntas) == preguntas["id"].max())

filas                       : 151
nulos                       : 0
preguntas duplicadas        : 0
con parentesis sobreviviente: 0
con espacio al borde        : 0
con doble espacio interno   : 0
vacias                      : 0
id de 1 a 151 | sin huecos: True


In [16]:
destino = SALIDA / "preguntas.csv"
preguntas.to_csv(destino, index=False, encoding="utf-8")

control = pd.read_csv(destino)
print("escrito:", destino, "|", control.shape)
control.head()

escrito: salida/preguntas.csv | (151, 2)


,id,pregunta
0,1,"Adjusted net enrollment rate, primary"
1,2,"Adjusted net enrollment rate, primary, female"
2,3,"Adjusted net enrollment rate, primary, male"
3,4,Adolescents out of school
4,5,"Adolescents out of school, female"


De los 164 indicadores únicos, 21 no tienen paréntesis, 111 tienen uno y 32 tienen dos.
Ninguno tiene paréntesis anidados.

Usé el patrón `\s*\([^()]*\)`, que matchea un paréntesis sin otro adentro e incluye el
espacio previo. Descarté la versión con `.*` porque el cuantificador es codicioso:
matchea desde el primer `(` hasta el último `)`, así que cuando hay texto entre los dos
grupos se lo lleva puesto. Son 5 casos, todos del tipo `School enrollment, primary
(gross), gender parity index (GPI)`, donde el codicioso devuelve `School enrollment,
primary` y borra `gender parity index`, que nunca estuvo entre paréntesis. En los otros
27 nombres con dos paréntesis los dos grupos van pegados al final, así que el codicioso
da el mismo resultado y el error no se nota. Incluir el espacio previo en el patrón me
ahorra tener que limpiar dobles espacios internos después: al trim final solo le quedan
los bordes.

Al sacar los paréntesis, 26 indicadores colapsan en 13 nombres repetidos y los 164 quedan
en 151. No son sinónimos. `School enrollment, primary (% gross)` y `School enrollment,
primary (% net)` son la tasa bruta y la neta, que miden cosas distintas, y después de la
limpieza quedan indistinguibles. Consolido igual porque la consigna pide indicadores
únicos sin duplicados, pero dejo la tabla de colisiones a la vista en el notebook para
que quede registrado qué se fundió con qué. Por lo mismo el CSV no lleva el
`Indicator Code`: después de consolidar, una pregunta corresponde a dos códigos. El `id`
lo asigno después de ordenar alfabéticamente, igual que en países.

Verifico que en las 151 preguntas no sobreviva ningún paréntesis, que no queden espacios
en los bordes ni dobles adentro, y que ninguna quede vacía. Ese chequeo fallaría si algún
indicador tuviera paréntesis anidados o uno abierto sin cerrar. Acá da limpio, pero lo
dejo porque el archivo lo publica un tercero y el formato puede cambiar.

## 3. Salida C — Duración de la educación primaria (2023)
> **CSV de Educación primaria en años**, ordenado por duración de mayor a menor según el
> año 2023. Campos requeridos: País / Código de País, duración de la educación primaria
> en años. Transformaciones esperadas: filtrar exclusivamente el indicador correspondiente
> a la duración de la educación primaria en años; considerar únicamente la medición
> correspondiente al año 2023; tratar valores nulos o no reportados para dicho año
> (especificar en el informe la estrategia adoptada: imputación, descarte, etc.);
> ordenar de mayor a menor según la duración en años.

In [17]:
# Filtro por el código y no por el nombre: el código es estable, el nombre puede cambiar
# de redacción entre ediciones del dataset.
INDICADOR = "SE.PRM.DURS"
ANIO = "2023"

dur = wb[wb["Indicator Code"] == INDICADOR].copy()

print("indicador:", dur["Indicator Name"].unique()[0])
print("filas:", len(dur))
print(f"con dato en {ANIO}: {dur[ANIO].notna().sum()}  |  sin dato: {dur[ANIO].isna().sum()}")
print(f"cobertura {ANIO} de este indicador: {100 * dur[ANIO].notna().mean():.1f} %")

indicador: Primary education, duration (years)
filas: 266
con dato en 2023: 255  |  sin dato: 11
cobertura 2023 de este indicador: 95.9 %


In [18]:
# Antes de decidir qué hago con los faltantes necesito saber si tienen historia. No es lo
# mismo un país que dejó de reportar un año que uno que nunca reportó nada.
sin_dato = dur[dur[ANIO].isna()].copy()
sin_dato["años_con_dato"] = dur[ANIOS].notna().sum(axis=1)
sin_dato["ultimo_año"] = dur[ANIOS].apply(lambda f: f.last_valid_index(), axis=1)

sin_dato[["Country Name", "Country Code", "años_con_dato", "ultimo_año"]]

,Country Name,Country Code,años_con_dato,ultimo_año
462,Afghanistan,AFG,53,2022
6366,Channel Islands,CHI,0,NaN
8170,Caribbean small states,CSS,0,NaN
12926,Faroe Islands,FRO,0,NaN
15058,Greenland,GRL,0,NaN
17846,Isle of Man,IMN,0,NaN
18174,Not classified,INX,0,NaN
24242,St. Martin (French part),MAF,0,NaN
27030,Northern Mariana Islands,MNP,0,NaN
32442,Pacific island small states,PSS,0,NaN


In [19]:
# Arrastro el último valor reportado hacia adelante sobre las columnas de año. Quien tiene
# serie hereda su último dato; quien nunca reportó queda en NaN y se descarta después.
# Guardo de qué año salió cada valor para que la imputación sea auditable.
dur["duracion"] = dur[ANIOS].ffill(axis=1)[ANIO]
dur["año_origen"] = dur[ANIOS].apply(lambda f: f.last_valid_index(), axis=1)

imputadas = dur[dur[ANIO].isna() & dur["duracion"].notna()]
print("filas imputadas:", len(imputadas))
print(imputadas[["Country Name", "Country Code", "duracion", "año_origen"]].to_string(index=False))

descartadas = dur[dur["duracion"].isna()]
print()
print("descartadas por no tener ningun dato en 64 años:", len(descartadas))
print(", ".join(descartadas["Country Name"]))

filas imputadas: 1
Country Name Country Code  duracion año_origen
 Afghanistan          AFG       6.0       2022

descartadas por no tener ningun dato en 64 años: 10
Channel Islands, Caribbean small states, Faroe Islands, Greenland, Isle of Man, Not classified, St. Martin (French part), Northern Mariana Islands, Pacific island small states, Kosovo


In [20]:
primaria = (
    dur.dropna(subset=["duracion"])
       .loc[:, ["Country Name", "Country Code", "duracion"]]
       .rename(columns={"Country Name": "nombre_pais",
                        "Country Code": "codigo_pais",
                        "duracion": "duracion_primaria_anios"})
)

# Los valores son 4 a 8, todos enteros. Sin el cast el CSV sale con "6.0".
primaria["duracion_primaria_anios"] = primaria["duracion_primaria_anios"].astype(int)

# 167 países empatan en 6 años. Desempato por nombre para que el orden sea reproducible
# y coincida con el que voy a armar en Hop.
primaria = (
    primaria
    .sort_values(["duracion_primaria_anios", "nombre_pais"],
                 ascending=[False, True], kind="stable")
    .reset_index(drop=True)
)

print(primaria.shape)
primaria.head(8)

(256, 3)


,nombre_pais,codigo_pais,duracion_primaria_anios
0,Guam,GUM,8
1,Ireland,IRL,8
2,Antigua and Barbuda,ATG,7
3,Australia,AUS,7
4,Bhutan,BTN,7
5,Botswana,BWA,7
6,British Virgin Islands,VGB,7
7,Denmark,DNK,7


In [21]:
print("filas            :", len(primaria))
print("nulos            :", int(primaria.isna().sum().sum()))
print("paises duplicados:", int(primaria["codigo_pais"].duplicated().sum()))
print("orden descendente:", primaria["duracion_primaria_anios"].is_monotonic_decreasing)
print("rango de valores :", primaria["duracion_primaria_anios"].min(),
      "a", primaria["duracion_primaria_anios"].max())
print()
print("cuantos por duracion:")
print(primaria["duracion_primaria_anios"].value_counts().sort_index(ascending=False).to_string())

filas            : 256
nulos            : 0
paises duplicados: 0
orden descendente: True
rango de valores : 4 a 8

cuantos por duracion:
duracion_primaria_anios
8      2
7     24
6    167
5     35
4     28


In [22]:
destino = SALIDA / "primaria_2023.csv"
primaria.to_csv(destino, index=False, encoding="utf-8")

control = pd.read_csv(destino)
print("escrito:", destino, "|", control.shape)
control.head()

escrito: salida/primaria_2023.csv | (256, 3)


,nombre_pais,codigo_pais,duracion_primaria_anios
0,Guam,GUM,8
1,Ireland,IRL,8
2,Antigua and Barbuda,ATG,7
3,Australia,AUS,7
4,Bhutan,BTN,7


Filtro por `Indicator Code` y no por el nombre del indicador, porque el código es estable
entre ediciones del dataset mientras que la redacción del nombre puede cambiar. Quedan
las 266 filas de `SE.PRM.DURS`, una por país o entidad. De esas, 255 tienen dato en 2023
y 11 no: una cobertura del 95,9 %, muy por encima del 8,8 % que tiene el archivo entero
para ese año. Tiene sentido, porque la duración de la primaria es un dato estructural que
fija la ley educativa de cada país y no hay que relevarlo con una encuesta como pasa con
una tasa de matriculación.

Los 11 faltantes no son todos iguales, así que no los trato igual. Diez no tienen ningún
dato en los 64 años de la serie, y varios ni siquiera son países (`Not classified`,
`Caribbean small states`, `Pacific island small states`). A esos los descarto. Imputarlos
con la moda me haría acertar seguido, pero acertar no es tener el dato, y para esas
entidades el valor no existe ni en concepto. Afganistán es el único caso distinto: tiene
53 años de serie y reportó 6 en 2022, así que le arrastro ese último valor. Quedan 256
filas. Guardo en `año_origen` de qué año salió cada valor para que la imputación sea
auditable. La limitación del arrastre es que no mira qué tan viejo es el dato que copia:
acá el único afectado es Afganistán con un hueco de un año, pero si hubiera un país cuyo
último reporte fuera de 1975 lo imputaría igual sin avisar.

Ordeno de mayor a menor por duración y desempato por nombre. El desempate no es
cosmético: 167 de las 256 filas valen 6 años, así que sin un segundo criterio el orden
dentro de ese bloque queda al azar y no lo voy a poder reproducir en Hop. Los valores van
de 4 a 8 y son todos enteros, así que los casteo para que el CSV no salga con "6.0".

Dejé los agregados regionales en el ranking. `World` aparece con 6 años, pero ese 6 es un
número que el World Bank calcula con un método que el CSV no documenta, y no corresponde
a ningún sistema educativo real. No es el promedio de los países: el promedio simple de
la columna da 5,75. Comparar ese 6 con el de Irlanda o el de Guam es comparar cosas de
distinto tipo. Lo asumo como limitación de esta salida, porque la consigna no pide
filtrarlos y separarlos pediría escribir a mano la lista de códigos de agregado.

## 4. Verificación de las salidas
Esta sección no resuelve ninguna consigna. Relee los tres CSV desde disco y los verifica,
y deja armada la función que voy a usar para comparar estas salidas contra las que genere
Apache Hop.

In [23]:
def revisar(nombre, columnas, clave):
    # Leo desde el archivo y no uso el DataFrame que tengo en memoria: lo que hay que
    # validar es el CSV que voy a entregar, no el objeto con el que lo generé.
    df = pd.read_csv(SALIDA / nombre)
    problemas = []

    if list(df.columns) != columnas:
        problemas.append(f"columnas {list(df.columns)} en vez de {columnas}")
    if df.isna().any().any():
        problemas.append(f"{int(df.isna().sum().sum())} nulos")
    if df[clave].duplicated().any():
        problemas.append(f"{int(df[clave].duplicated().sum())} duplicados en {clave}")
    if "id" in df.columns and df["id"].tolist() != list(range(1, len(df) + 1)):
        problemas.append("el id no es 1..n correlativo")

    return df, {"archivo": nombre, "filas": len(df), "columnas": len(df.columns),
                "estado": "OK" if not problemas else "; ".join(problemas)}


paises,    r1 = revisar("paises.csv",        ["id", "nombre_pais", "codigo_pais"], "codigo_pais")
preguntas, r2 = revisar("preguntas.csv",     ["id", "pregunta"],                   "pregunta")
primaria,  r3 = revisar("primaria_2023.csv",
                        ["nombre_pais", "codigo_pais", "duracion_primaria_anios"], "codigo_pais")

pd.DataFrame([r1, r2, r3])

,archivo,filas,columnas,estado
0,paises.csv,266,3,OK
1,preguntas.csv,151,2,OK
2,primaria_2023.csv,256,3,OK


In [24]:
# Las tres salidas vienen del mismo archivo, así que tienen que ser consistentes entre sí.
# Si no lo fueran, algo se me desalineó en el camino.
faltan = set(primaria["codigo_pais"]) - set(paises["codigo_pais"])
print("codigos de primaria que no estan en paises:", len(faltan), faltan or "")

cruce = primaria.merge(paises, on="codigo_pais", suffixes=("_prim", "_pais"))
print("codigos con nombre distinto entre los dos archivos:",
      int((cruce["nombre_pais_prim"] != cruce["nombre_pais_pais"]).sum()))

fuera = paises[~paises["codigo_pais"].isin(primaria["codigo_pais"])]
print("paises que quedaron fuera del ranking:", len(fuera))
print("  ", ", ".join(fuera["nombre_pais"]))

print("el indicador de la salida C esta en preguntas.csv:",
      "Primary education, duration" in set(preguntas["pregunta"]))

codigos de primaria que no estan en paises: 0 
codigos con nombre distinto entre los dos archivos: 0
paises que quedaron fuera del ranking: 10
   Caribbean small states, Channel Islands, Faroe Islands, Greenland, Isle of Man, Kosovo, Northern Mariana Islands, Not classified, Pacific island small states, St. Martin (French part)
el indicador de la salida C esta en preguntas.csv: True


In [25]:
def comparar(ruta_a, ruta_b, etiqueta_a="pandas", etiqueta_b="hop"):
    a = pd.read_csv(ruta_a)
    b = pd.read_csv(ruta_b)
    print(f"{etiqueta_a}: {a.shape}   {etiqueta_b}: {b.shape}")

    if list(a.columns) != list(b.columns):
        print("  DISTINTAS COLUMNAS")
        print("   ", etiqueta_a, list(a.columns))
        print("   ", etiqueta_b, list(b.columns))
        return False

    if a.equals(b):
        print("  IDENTICOS: mismo contenido y mismo orden")
        return True

    # Comparo como conjuntos de filas y no posición por posición: si a un archivo le falta
    # una fila, la comparación posicional desalinea todo lo que sigue y reporta cientos de
    # diferencias que no existen.
    cruce = a.merge(b, how="outer", on=list(a.columns), indicator=True)
    solo_a = cruce[cruce["_merge"] == "left_only"].drop(columns="_merge")
    solo_b = cruce[cruce["_merge"] == "right_only"].drop(columns="_merge")

    if solo_a.empty and solo_b.empty:
        print("  MISMO CONTENIDO, DISTINTO ORDEN")
        return False

    print(f"  filas solo en {etiqueta_a}: {len(solo_a)}")
    if len(solo_a):
        print(solo_a.head(5).to_string(index=False))
    print(f"  filas solo en {etiqueta_b}: {len(solo_b)}")
    if len(solo_b):
        print(solo_b.head(5).to_string(index=False))
    return False

In [26]:
# Todavía no tengo las salidas de Hop, así que pruebo la función contra casos que armo yo.
# Una función de verificación en la que no verifiqué nada no sirve de mucho.
import tempfile
tmp = Path(tempfile.mkdtemp())

print("-- contra si mismo, tiene que dar IDENTICOS:")
comparar(SALIDA / "paises.csv", SALIDA / "paises.csv", "a", "b")

print()
print("-- mismo contenido en otro orden:")
pd.read_csv(SALIDA / "paises.csv").iloc[::-1].to_csv(tmp / "revertido.csv", index=False)
comparar(SALIDA / "paises.csv", tmp / "revertido.csv", "real", "revertido")

print()
print("-- con un valor cambiado y una fila borrada:")
t = pd.read_csv(SALIDA / "paises.csv")
t.loc[5, "nombre_pais"] = "XXXX"
t.drop(index=10).to_csv(tmp / "trucho.csv", index=False)
comparar(SALIDA / "paises.csv", tmp / "trucho.csv", "real", "trucho")

-- contra si mismo, tiene que dar IDENTICOS:
a: (266, 3)   b: (266, 3)
  IDENTICOS: mismo contenido y mismo orden

-- mismo contenido en otro orden:
real: (266, 3)   revertido: (266, 3)
  MISMO CONTENIDO, DISTINTO ORDEN

-- con un valor cambiado y una fila borrada:
real: (266, 3)   trucho: (265, 3)
  filas solo en real: 2
 id    nombre_pais codigo_pais
  6 American Samoa         ASM
 11      Argentina         ARG
  filas solo en trucho: 1
 id nombre_pais codigo_pais
  6        XXXX         ASM


False

Releo los tres CSV desde disco en vez de usar los DataFrames que tengo en memoria. Lo que
hay que validar es el archivo que entrego, no el objeto con el que lo generé. Los tres
pasan: 266 filas en `paises.csv`, 151 en `preguntas.csv` y 256 en `primaria_2023.csv`,
sin nulos, sin duplicados en la clave y con los `id` correlativos de 1 a n.

Las tres salidas vienen del mismo archivo, así que también las cruzo entre sí. Los 256
códigos de `primaria_2023` están todos en `paises.csv` y con el mismo nombre asociado.
Ese chequeo caza un error puntual: si en algún paso un `sort` o un `merge` hubiera
reordenado una columna sin la otra, la correspondencia entre nombre y código se habría
roto y acá saltaría. Los 10 países que quedan fuera del ranking son exactamente los que
descarté en la sección 3 por no tener ningún dato en 64 años.

`comparar()` compara contenido y no formato. Hop y pandas casi seguro van a escribir el
mismo dato con otras comillas o con otro salto de línea, y no quiero que la comparación
falle por eso. Compara conjuntos de filas en lugar de posición por posición, porque si a
un archivo le falta una fila la comparación posicional desalinea todo lo que sigue y
reporta cientos de diferencias que no existen. El orden sí cuenta como diferencia: en
`paises` y `preguntas` el orden es lo que define el `id`, así que otro orden es otro
archivo, y en `primaria_2023` el ordenamiento es parte de lo que pide la consigna.

Probé la función antes de confiar en ella, contra un archivo igual, uno con el orden
invertido y uno con una fila borrada y un valor cambiado. Le quedan tres puntos ciegos
para tener presentes cuando compare contra Hop: no matchea nulos contra nulos, colapsa
filas duplicadas idénticas, y no ve las diferencias de formato que cambian el archivo sin
cambiar los datos, como un BOM al principio o el estilo de comillas. Ninguna me afecta con
estas salidas, pero le pueden romper la lectura a otra herramienta.

In [27]:
# Ahora si, contra las salidas reales de Apache Hop.
HOP = Path("salida-hop")

for nombre in ["paises.csv", "preguntas.csv", "primaria_2023.csv"]:
    print(f"== {nombre}")
    comparar(SALIDA / nombre, HOP / nombre)
    print()


== paises.csv
pandas: (266, 3)   hop: (266, 3)
  IDENTICOS: mismo contenido y mismo orden

== preguntas.csv
pandas: (151, 2)   hop: (151, 2)
  IDENTICOS: mismo contenido y mismo orden

== primaria_2023.csv
pandas: (256, 3)   hop: (256, 3)
  IDENTICOS: mismo contenido y mismo orden

